In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS jarvis_etl.bronze;
CREATE SCHEMA IF NOT EXISTS jarvis_etl.silver;
CREATE SCHEMA IF NOT EXISTS jarvis_etl.gold;

In [0]:
%skip
# skip this cell 
jdbc_url = (
    "jdbc:sqlserver://jarvis-etl-sqlserver.database.windows.net:1433;"
    "database=jarvis-etl-sqldb;"
    "encrypt=true;"
    "trustServerCertificate=false;"
    "hostNameInCertificate=*.database.windows.net;"
    "loginTimeout=30;"
)



transactions_df = (
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "dbo.transactions_data")
    .option("user", user)
    .option("password", password)
    .option("fetchsize", "10000")
    .load()
)

transactions_df.write.mode("overwrite").saveAsTable("jarvis_etl.bronze.transactions")
print(f"transactions row count: {transactions_df.count()}")


transactions row count: 15857925


In [0]:
from pyspark.sql.functions import col, regexp_replace, trim
from pyspark.sql.types import DecimalType

transactions_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load("abfss://labdata@jarvisetlstorage.dfs.core.windows.net/transactions_data.csv")
)

transactions_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("jarvis_etl.bronze.transactions")
print(f"transactions row count: {transactions_df.count()}")

transactions row count: 13305915


In [0]:
%sql 
Select * from jarvis_etl.bronze.transactions where id = '7475501';

id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors
7475501,2010-01-01T04:31:00.000Z,526,5917,$22.23,Online Transaction,16798,ONLINE,null,null,4121,null


In [0]:
jdbc_url = (
    "jdbc:sqlserver://jarvis-etl-sqlserver.database.windows.net:1433;"
    "database=jarvis-etl-sqldb;"
    "encrypt=true;"
    "trustServerCertificate=false;"
    "hostNameInCertificate=*.database.windows.net;"
    "loginTimeout=30;"
)


cards_df = (
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "dbo.cards_data")
    .option("user", user)
    .option("password", password)
    .option("fetchsize", "10000")
    .load()
)

cards_df.write.mode("overwrite").saveAsTable("jarvis_etl.bronze.cards")
print(f"cards row count: {cards_df.count()}")

cards row count: 6146


In [0]:
users_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load("abfss://labdata@jarvisetlstorage.dfs.core.windows.net/users_data.csv")
)

users_df.write.mode("overwrite").saveAsTable("jarvis_etl.bronze.users")
print(f"users row count: {users_df.count()}")

users row count: 2000


In [0]:
mcc_raw_df = (
    spark.read
    .format("json")
    .option("multiLine", "true")
    .load("abfss://labdata@jarvisetlstorage.dfs.core.windows.net/mcc_codes.json")
)

from pyspark.sql.functions import explode, create_map, lit, col
from pyspark.sql import functions as F

mcc_df = mcc_raw_df.select(
    explode(
        create_map(*[x for col_name in mcc_raw_df.columns for x in (lit(col_name), col(f"`{col_name}`"))])
    ).alias("mcc_code", "description")
)

mcc_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("jarvis_etl.bronze.mcc_codes")
print(f"mcc_codes row count: {mcc_df.count()}")

mcc_codes row count: 109


In [0]:
%sql
SELECT * FROM jarvis_etl.bronze.mcc_codes LIMIT 10;

mcc_code,description
1711,"Heating, Plumbing, Air Conditioning Contractors"
3000,Steelworks
3001,Steel Products Manufacturing
3005,Miscellaneous Metal Fabrication
3006,Miscellaneous Fabricated Metal Products
3007,Coated and Laminated Products
3008,Steel Drums and Barrels
3009,Fabricated Structural Metal Products
3058,"Tools, Parts, Supplies Manufacturing"
3066,Miscellaneous Metals


In [0]:
fraud_df = (
    spark.read
    .format("json")
    .load("abfss://labdata@jarvisetlstorage.dfs.core.windows.net/train_fraud_labels.jsonl")
)

fraud_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("jarvis_etl.bronze.fraud_labels")
print(f"fraud_labels row count: {fraud_df.count()}")

fraud_labels row count: 8914963


In [0]:
%sql
OPTIMIZE jarvis_etl.bronze.transactions ZORDER BY (id);
OPTIMIZE jarvis_etl.bronze.cards ZORDER BY (id);
OPTIMIZE jarvis_etl.bronze.users ZORDER BY (id);
OPTIMIZE jarvis_etl.bronze.mcc_codes;
OPTIMIZE jarvis_etl.bronze.fraud_labels;

In [0]:
%sql
SELECT 'transactions' as table_name, COUNT(*) as row_count FROM jarvis_etl.bronze.transactions
UNION ALL
SELECT 'cards', COUNT(*) FROM jarvis_etl.bronze.cards
UNION ALL
SELECT 'users', COUNT(*) FROM jarvis_etl.bronze.users
UNION ALL
SELECT 'mcc_codes', COUNT(*) FROM jarvis_etl.bronze.mcc_codes
UNION ALL
SELECT 'fraud_labels', COUNT(*) FROM jarvis_etl.bronze.fraud_labels;

table_name,row_count
transactions,15857925
cards,6146
users,2000
mcc_codes,109
fraud_labels,8914963
